# 스태커 크레인 S자 적재 (Factory IO / Modbus TCP / pymodbus 3.x)
계획서: `StackerCrane_S_PLAN.md`. 셀을 위에서부터 차례로 실행한다.
- Factory IO 없이 시험: `python fake_factoryio_server.py --port 5020 --auto-start` 실행 후 1번 셀에서 `HOST='127.0.0.1'`, `PORT=5020`

## 1. 설정

In [ ]:
import os, time

HOST = os.environ.get('FIO_HOST', '192.168.0.16')   # Factory IO PC IP (같은 PC면 127.0.0.1)
PORT = int(os.environ.get('FIO_PORT', '502'))
DEVICE_ID = 1

HOME_POS = 99               # 정위치(복귀) Target Position
POLL = 0.05                 # 입력 폴링 주기 [s]
EMIT_PULSE = 0.3            # Emitter ON 유지 시간 [s]
LIFT_WAIT = 1.0             # Lift ON/OFF 후 대기 [s]
MOVE_START_TIMEOUT = 1.5    # 위치 지령 후 Moving 신호 시작 대기 [s]
STEP_TIMEOUT = 30           # 각 단계 최대 대기 [s]
MOVE_TIMEOUT = 60           # 이동 완료 최대 대기 [s]
WAIT_START_EACH_CYCLE = False   # True면 매 사이클 Start 버튼 대기
START_INDEX = 0             # 중단 후 이어서 할 때 시작 순번(0부터)
SENSOR_ACTIVE_LOW = True    # Factory IO 'At Load'(반사형 센서): 물건 있으면 0 → 코드에서 반전
STOP_FEED_ON_SENSOR = True  # 센서 감지 시 Conveyor0도 정지. False면 원래 사양(계속 ON)인데,
                            # Factory IO에서는 뒤 팔레트가 밀려와 크레인에 걸림(실측)
SETTLE = 0.5                # Load in 정지 후 물건 안정화 대기 [s]

## 2. 주소 맵 (Modbus 번호 → pymodbus 0-based 주소)

In [ ]:
DI = {                       # Discrete Input 10001~
    'Sensor': 0, 'MovingX': 1, 'MovingZ': 2,
    'LeftLimit': 3, 'RightLimit': 4, 'MiddleLimit': 5,
    'Start': 6, 'Reset': 7,
}
COIL = {                     # Coil 00001~
    'Lift': 0, 'ForkLeft': 1, 'ForkRight': 2, 'LoadIn': 3, 'LoadOut': 4,
    'Conveyor0': 8, 'Conveyor1': 9, 'Emitter0': 10, 'Remover0': 11,
}
HR_TARGET = 0                # Holding Register 40001

FORK_PICK_SIDE = 'ForkLeft'      # Load in(픽업) 쪽 포크
FORK_PLACE_SIDE = 'ForkRight'    # 랙(적재) 쪽 포크
PICK_LIMIT = 'LeftLimit'
PLACE_LIMIT = 'RightLimit'

## 3. 접속

In [ ]:
from pymodbus.client import ModbusTcpClient

try:
    client.close()           # 셀 재실행 시 이전 연결 정리
except NameError:
    pass

client = ModbusTcpClient(HOST, port=PORT, timeout=3)
if not client.connect():
    raise ConnectionError(f'Modbus TCP 연결 실패: {HOST}:{PORT}')
print(f'Modbus TCP 연결 성공: {HOST}:{PORT} (device_id={DEVICE_ID})')

## 4. 헬퍼 함수

In [ ]:
class AbortSequence(Exception):
    """Reset 버튼으로 중단"""

def read_inputs():
    rr = client.read_discrete_inputs(0, count=8, device_id=DEVICE_ID)
    if rr.isError():
        raise RuntimeError(f'DI 읽기 실패: {rr}')
    bits = rr.bits[:8]
    s = {name: bool(bits[addr]) for name, addr in DI.items()}
    if SENSOR_ACTIVE_LOW:
        s['Sensor'] = not s['Sensor']   # 이후 코드에서는 Sensor=True 가 '물건 있음'
    return s

def set_coil(name, value):
    wr = client.write_coil(COIL[name], bool(value), device_id=DEVICE_ID)
    if wr.isError():
        raise RuntimeError(f'코일 쓰기 실패 {name}: {wr}')

def set_target(pos):
    wr = client.write_register(HR_TARGET, int(pos), device_id=DEVICE_ID)
    if wr.isError():
        raise RuntimeError(f'Target Position 쓰기 실패 {pos}: {wr}')

def all_off():
    for name in COIL:
        set_coil(name, False)

def wait_until(cond, step, timeout=STEP_TIMEOUT, check_reset=True):
    """cond(inputs)->bool 이 True가 될 때까지 대기. Reset/타임아웃 시 예외."""
    t_end = time.time() + timeout
    while True:
        s = read_inputs()
        if check_reset and s['Reset']:
            raise AbortSequence(f'Reset 버튼 눌림 ({step})')
        if cond(s):
            return s
        if time.time() > t_end:
            raise TimeoutError(f'{step}: {timeout}s 초과')
        time.sleep(POLL)

def wait_move_done(step):
    # (a) Moving 신호가 켜질 때까지 잠깐 대기 (이미 그 위치면 안 켜질 수 있음)
    try:
        wait_until(lambda s: s['MovingX'] or s['MovingZ'], step + ' 이동 시작', MOVE_START_TIMEOUT)
    except TimeoutError:
        pass
    # (b) MovingX, MovingZ 모두 OFF
    wait_until(lambda s: not s['MovingX'] and not s['MovingZ'], step + ' 이동 완료', MOVE_TIMEOUT)
    time.sleep(0.2)          # (c) 안정화

def wait_start_button():
    print('Start 버튼 대기...')
    wait_until(lambda s: not s['Start'], 'Start 해제 대기', timeout=10**9)   # 상승엣지 검출
    wait_until(lambda s: s['Start'], 'Start 대기', timeout=10**9)
    print('Start!')

## 5. S 패턴 → Target Position 리스트

In [ ]:
COLS, ROWS = 9, 6

# 정면에서 본 모양, 위(z=5) → 아래(z=0), 왼쪽 → 오른쪽.  '#' = 적재
S_PATTERN = [
    '.#######.',
    '##.......',
    '.#######.',
    '.......##',
    '##.....##',
    '.#######.',
]

def pos_from_cell(z, c):
    """z: 0=맨 아래, c: 0=맨 오른쪽.  우측 아래=1, 좌측 위=54"""
    return z * COLS + c + 1

def pattern_to_positions(pattern):
    assert len(pattern) == ROWS and all(len(r) == COLS for r in pattern)
    out = []
    for i, row in enumerate(pattern):
        z = ROWS - 1 - i
        for v, ch in enumerate(row):          # v: 0=왼쪽
            if ch == '#':
                out.append(pos_from_cell(z, COLS - 1 - v))
    return sorted(out)                        # 아래 행부터, 오른쪽→왼쪽 순서로 적재

POSITIONS = pattern_to_positions(S_PATTERN)
assert POSITIONS == [2, 3, 4, 5, 6, 7, 8, 10, 11, 17, 18, 19, 20, 29, 30, 31, 32, 33, 34, 35,
                     44, 45, 47, 48, 49, 50, 51, 52, 53], POSITIONS

print(f'적재 개수: {len(POSITIONS)}')
print(POSITIONS)
for i, row in enumerate(S_PATTERN):
    z = ROWS - 1 - i
    print(f'z={z} ' + ' '.join(f'{pos_from_cell(z, COLS-1-v):2d}' if ch == '#' else ' .' for v, ch in enumerate(row)))

## 6. IO 점검 (선택, 처음 한 번)
- 입력 상태 확인. 위치 번호 규칙 확인은 아래 주석을 풀고 실행해 크레인 위치를 눈으로 본다.
  - `9` → 맨 아래 행 맨 왼쪽, `10` → 아래서 2번째 행 맨 오른쪽 이어야 함. 다르면 `pos_from_cell()` 수정.

In [ ]:
print(read_inputs())
# set_target(9);  wait_move_done('테스트 9')
# set_target(10); wait_move_done('테스트 10')
# set_target(HOME_POS); wait_move_done('테스트 복귀')

## 7. 1사이클 시퀀스

In [ ]:
def run_cycle(pos):
    # [1] 공급 시작 (Load in 포함 유지) + Emitter 1회
    for name in ('Conveyor0', 'Conveyor1', 'Remover0', 'LoadIn'):
        set_coil(name, True)
    set_coil('Emitter0', True); time.sleep(EMIT_PULSE); set_coil('Emitter0', False)

    # [2] 센서 감지 → Load in 정지 (계속 돌리면 물건이 컨베이어 끝에서 떨어짐)
    wait_until(lambda s: s['Sensor'], '[2] Sensor 감지')
    set_coil('LoadIn', False)
    if STOP_FEED_ON_SENSOR:
        set_coil('Conveyor0', False)    # 다음 사이클 [1]에서 다시 ON
    time.sleep(SETTLE)

    # [3] 픽업
    set_coil(FORK_PICK_SIDE, True)
    wait_until(lambda s: s[PICK_LIMIT], '[3] 픽업 포크 Limit')
    set_coil('Lift', True); time.sleep(LIFT_WAIT)
    set_coil(FORK_PICK_SIDE, False)
    wait_until(lambda s: s['MiddleLimit'], '[3] Middle Limit')

    # [4] 적재 위치로 이동
    set_target(pos)
    wait_move_done(f'[4] pos={pos}')

    # [5] 내려놓기 → 복귀 지령
    set_coil(FORK_PLACE_SIDE, True)
    wait_until(lambda s: s[PLACE_LIMIT], '[5] 적재 포크 Limit')
    set_coil('Lift', False); time.sleep(LIFT_WAIT)
    set_coil(FORK_PLACE_SIDE, False)
    wait_until(lambda s: s['MiddleLimit'], '[5] Middle Limit')
    set_target(HOME_POS)

    # [6] 복귀 완료
    wait_move_done('[6] 복귀')

## 8. 실행 (중단: ■ 버튼 또는 Factory IO Reset 버튼)

In [ ]:
done_index = START_INDEX
try:
    all_off()
    set_target(HOME_POS)
    wait_move_done('초기 복귀')
    wait_start_button()
    for i in range(START_INDEX, len(POSITIONS)):
        if WAIT_START_EACH_CYCLE and i > START_INDEX:
            wait_start_button()
        pos = POSITIONS[i]
        t0 = time.time()
        run_cycle(pos)
        done_index = i + 1
        print(f'[{done_index}/{len(POSITIONS)}] pos={pos} 적재 완료 ({time.time()-t0:.1f}s)')
    print('S자 적재 완료')
except AbortSequence as e:
    print('중단:', e)
except KeyboardInterrupt:
    print('사용자 중단')
except Exception as e:
    print('오류:', repr(e))
    raise
finally:
    all_off()
    print(f'모든 코일 OFF. 다음에 이어서 하려면 START_INDEX = {done_index}')

## 9. 연결 종료

In [ ]:
client.close()
print('연결 종료')